# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [1]:
%load_ext dotenv
%dotenv ../05_src/.secrets

In [2]:
import sys
sys.path.append('../../05_src/')

## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

file_path = "https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf"
loader = PyPDFLoader(file_path)

docs_load= loader.load()

print(len(docs_load))

docs = ""
for page in docs_load:
    docs += page.page_content + "\n"
#print(docs[:5000])



13


## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [16]:
model_tone ="Formal Academic Writing"

In [17]:
instruction = f"""
                "You are a technical analyst. Generate a structured response based on the provided text.
              "The 'summary' field must be written strictly in the following tone: {model_tone}. 
              "The 'relevance' field should focus on the professional development of an AI Engineer.
        """

In [18]:
context = f"""
        "Analyze the following article content: {docs}. "
        "Focus on the core concepts of identifying strengths, working styles, and values."
   """ 


In [20]:
from openai import OpenAI
from pydantic import BaseModel,Field
import os

client = OpenAI(default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')},
    base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1')


class generateSummaryPDF(BaseModel):
    author: str = Field(description="The author of the article.")
    title: str = Field(description="The title of the article.")
    relevance: str = Field(description="Statement on professional relevance for an AI professional.")
    summary: str = Field(description="A concise summary (max 1000 tokens).")
    

response = client.responses.parse(
    model="gpt-4o-mini",
    max_output_tokens=1000,
    input=[
            {   "role": "system", 
                "content": instruction.format(selected_tone=model_tone),
            },
            {
                "role": "user",
                "content": context.format(docs=docs),
            },
        ],  
    text_format=generateSummaryPDF
)

event = response.output_parsed


In [ ]:
event
summary_out=event.summary
summary_out

generateSummaryPDF(author='Peter F. Drucker', title='Managing Oneself', relevance='Understanding personal strengths, work styles, and values is imperative for AI professionals to innovate effectively and lead successful projects.', summary="In the contemporary knowledge economy, success is increasingly contingent upon self-awareness—specifically in understanding one's strengths, values, and working styles. Drucker posits that knowledge workers must assume the role of 'chief executive officer' of their careers, taking proactive steps to navigate their professional paths. The ability to accurately identify personal strengths is emphasized, advocating for 'feedback analysis' to ascertain one's competencies. Feedback analysis involves documenting expectations for key decisions and later comparing them with actual outcomes, facilitating an understanding of effective performance patterns. \n\nSimilarly, comprehension of how one learns and interacts with others is crucial; identifying whether

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

#Summarisation Metric

In [25]:
from deepeval import evaluate
from deepeval.test_case import LLMTestCase
from deepeval.metrics import SummarizationMetric
from deepeval.models import GPTModel

model = GPTModel(
    model="gpt-4o-mini",
    temperature=0,
    # api_key='any value',
    default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')},
    base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
)


In [ ]:

test_case = LLMTestCase(input=context.format(docs=docs)
                        , actual_output=summary_out)

summarisation_metric = SummarizationMetric(
    threshold=0.5,
    model=model,
    assessment_questions=[
        "Is the coverage score based on a percentage of 'yes' answers?",
        "Does the score ensure the summary's accuracy with the source?",
        "Does a higher score mean a more comprehensive summary?",
        "Is the summary concise and relevant to the professional development of an AI Engineer?",
        "Does the summary maintain a formal academic tone as specified in the instruction?"

    ]
)
# To run metric as a standalone
summarisation_metric.measure(test_case)

evaluate(test_cases=[test_case], metrics=[summarisation_metric])

print(f"""
      "Summarization Score": {summarisation_metric.score},
      "Summarization Reason": {summarisation_metric.reason}""")

#result_summarisation_metric.model_dump()

G-VAL Metric

Coherence

In [ ]:
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams
from deepeval.models import GPTModel



coherence_metric = GEval(
    name="Clarity",
    evaluation_steps=[
        "Evaluate whether the response uses clear and direct language.",
        "Check if the explanation avoids jargon or explains it when used.",
        "Assess whether complex ideas are presented in a way that's easy to follow.",
        "Identify any vague or confusing parts that reduce understanding.",
        "Vague language, or contradicting OPINIONS, are not OK"
    ],
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model,
)

test_case = LLMTestCase(
    input=context.format(docs=docs),
    actual_output=summary_out
)


result_coherence_metric = evaluate(test_cases=[test_case], metrics=[coherence_metric])

print(f"""
      "Coherence Score": {coherence_metric.score},
        "Coherence Reason": {coherence_metric.reason}""")

result_coherence_metric.model_dump()

✨ You're running DeepEval's latest Clarity [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

Output()



Metrics Summary

  - ✅ Clarity [GEval] (score: 0.8679178692681617, threshold: 0.5, strict: False, evaluation model: gpt-4o-mini, reason: The response uses clear and direct language, effectively summarizing Drucker's key concepts without excessive jargon. It presents complex ideas about self-awareness and career navigation in an accessible manner. However, while the explanation is mostly clear, it could benefit from slightly more straightforward phrasing in some areas to enhance understanding further., error: None)

For test case:

  - input: Analyze the following article content: www.hbr.org
B
 
EST  
 
OF  HBR 1999
 
Managing Oneself
 
by Peter F . Drucker
 
•
 
Included with this full-text 
 
Harvard Business Review
 
 article:
The Idea in Brief—the core idea
The Idea in Practice—putting the idea to work
 
1
 
Article Summary
 
2
 
Managing Oneself
A list of related materials, with annotations to guide further
exploration of the article’s ideas and applications
 
12
 
Further Readi

✓ Evaluation completed 🎉! (time taken: 10.46s | token cost: 0.0001401 USD)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» What to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

Tonality

In [ ]:
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams
from deepeval.models import GPTModel



tonality_metric = GEval(
    name="Professionalism",
    evaluation_steps=[
        "Determine whether the actual output maintains a professional tone throughout.",
        "Evaluate if the language in the actual output reflects expertise and domain-appropriate formality.",
        "Ensure the actual output stays contextually appropriate and avoids casual or ambiguous expressions.",
        "Check if the actual output is clear, respectful, and avoids slang or overly informal phrasing."
        "Vague language, or contradicting OPINIONS, are not OK"
    ],
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model,
)

test_case = LLMTestCase(
    input=context.format(docs=docs),
    actual_output=summary_out
)

result_tonality_metric = evaluate(test_cases=[test_case], metrics=[tonality_metric])

print(f"""
      "Tonality Score": {tonality_metric.score},
        "Tonality Reason": {tonality_metric.reason}""")

result_tonality_metric.model_dump()

Safety

In [ ]:
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams
from deepeval.models import GPTModel



safety_metric = GEval(
    name="PII Leakage",
    evaluation_steps=[
        "Determine whether the actual output maintains a professional tone throughout.",
        "Evaluate if the language in the actual output reflects expertise and domain-appropriate formality.",
        "Ensure the actual output stays contextually appropriate and avoids casual or ambiguous expressions.",
        "Check if the actual output is clear, respectful, and avoids slang or overly informal phrasing."
        "Vague language, or contradicting OPINIONS, are not OK"
    ],
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model,
)

test_case = LLMTestCase(
    input=context.format(docs=docs),
    actual_output=summary_out
)

result_safety_metric = evaluate(test_cases=[test_case], metrics=[safety_metric])

print(f"""
      "Safety Score": {safety_metric.score},
        "Safety Reason": {safety_metric.reason}""")

result_safety_metric.model_dump()

✨ You're running DeepEval's latest PII Leakage [GEval] Metric! (using gpt-4o-mini, strict=False, 
async_mode=True)...

Output()



Metrics Summary

  - ✅ PII Leakage [GEval] (score: 0.9754914981353654, threshold: 0.5, strict: False, evaluation model: gpt-4o-mini, reason: The actual output maintains a professional tone throughout, reflecting expertise in the subject matter. The language used is formal and appropriate for the context, avoiding any casual or ambiguous expressions. It clearly articulates the core concepts of the article while remaining respectful and free of slang, aligning perfectly with the evaluation steps., error: None)

For test case:

  - input: Analyze the following article content: www.hbr.org
B
 
EST  
 
OF  HBR 1999
 
Managing Oneself
 
by Peter F . Drucker
 
•
 
Included with this full-text 
 
Harvard Business Review
 
 article:
The Idea in Brief—the core idea
The Idea in Practice—putting the idea to work
 
1
 
Article Summary
 
2
 
Managing Oneself
A list of related materials, with annotations to guide further
exploration of the article’s ideas and applications
 
12
 
Further Reading
Suc

✓ Evaluation completed 🎉! (time taken: 7.24s | token cost: 0.00014145 USD)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» What to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.


      "Safety Score": None,
        "Safety Reason": None


{'test_results': [{'name': 'test_case_0',
   'success': True,
   'metrics_data': [{'name': 'PII Leakage [GEval]',
     'threshold': 0.5,
     'success': True,
     'score': 0.9754914981353654,
     'reason': 'The actual output maintains a professional tone throughout, reflecting expertise in the subject matter. The language used is formal and appropriate for the context, avoiding any casual or ambiguous expressions. It clearly articulates the core concepts of the article while remaining respectful and free of slang, aligning perfectly with the evaluation steps.',
     'strict_mode': False,
     'evaluation_model': 'gpt-4o-mini',
     'error': None,
     'evaluation_cost': 0.00014145,
     'verbose_logs': 'Criteria:\nNone \n \nEvaluation Steps:\n[\n    "Determine whether the actual output maintains a professional tone throughout.",\n    "Evaluate if the language in the actual output reflects expertise and domain-appropriate formality.",\n    "Ensure the actual output stays contextually 

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [86]:
print(safety_metric.reason)


None


In [95]:
promptNew = f""""
        There is summary evaluated via the following metrics: safety_metric.
        The summary  which was evaluated is {summary_out}      
        Lastly, the safety score is {safety_metric.score} and {safety_metric.reason}, evaluating the presence of any potentially harmful or inappropriate content in the summary. 
        While generating new summary you should consider the score and reason and provide the answer with respect to that   
"""

In [ ]:
from openai import OpenAI
from pydantic import BaseModel,Field
import os

client = OpenAI(default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')},
    base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1')

class generateSummaryPDFEnhanced(BaseModel):
    author: str = Field(description="The author of the article.")
    title: str = Field(description="The title of the article.")
    relevance: str = Field(description="Statement on professional relevance for an AI professional.")
    summary: str = Field(description="A concise summary (max 1000 tokens).")

response = client.responses.parse(
    model="gpt-4o-mini",
    max_output_tokens=1000,
    input=[
            {   "role": "system", 
                "content": promptNew.format(selected_tone=model_tone),
            },
            {
                "role": "user",
                "content": context.format(docs=docs)
            },
        ],  
    text_format=generateSummaryPDFEnhanced
)

result = response.output_parsed



In [100]:
summary_out_new=result.summary
summary_out_new

'In "Managing Oneself," Peter F. Drucker outlines the necessity of self-knowledge in achieving success in the contemporary knowledge economy. He argues that professionals need to identify their strengths, values, and preferred working styles to navigate their careers effectively. This self-awareness allows individuals to align their roles with their intrinsic capabilities, fostering both personal fulfillment and organizational productivity. Drucker provides practical methods for self-assessment and advocates for continuous learning and adaptation, emphasizing that individual effectiveness is closely tied to personal insight. The article serves as a guide for professionals to enhance their performance and adapt to changes in the workplace, highlighting the importance of knowing oneself in order to thrive.'

safety evaluation

In [ ]:
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams
from deepeval.models import GPTModel



safety_metricNew = GEval(
    name="PII Leakage",
    evaluation_steps=[
        "Determine whether the actual output maintains a professional tone throughout.",
        "Evaluate if the language in the actual output reflects expertise and domain-appropriate formality.",
        "Ensure the actual output stays contextually appropriate and avoids casual or ambiguous expressions.",
        "Check if the actual output is clear, respectful, and avoids slang or overly informal phrasing and better than previous summary"
        "Vague language, or contradicting OPINIONS, are not OK"
    ],
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model,
)

test_case = LLMTestCase(
    input=context.format(docs=docs),
    actual_output=summary_out_new)

result_safety_metricNew = evaluate(test_cases=[test_case], metrics=[safety_metricNew])

print(f"""
      "Safety Score": {safety_metricNew.score},
        "Safety Reason": {safety_metricNew.reason}""")

result_safety_metricNew.model_dump()

Please, do not forget to add your comments.

In [ ]:
""" The results are not actually wondering as score is almost same after evaluation.
This metric strategy is not fully effective as it is not able to capture the improvement in summary and also the reason provided by metric is not relevant to the summary.
It is just repeating the same reason for both summaries. 
We need more checkpoints to improve the metric and the results. """


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
